# Storage, Filesystems & Mounts

## What's covered

- **The storage stack** — the layered mental model from hardware up to files: device → partition → (optional LVM) → filesystem → mount point
- **Block devices** — what they are, how Linux names them (`/dev/sda`, `/dev/nvme0n1`, `/dev/vda`)
- **`lsblk`** — the one command you'll run constantly to see what's attached
- **Partitions** — MBR vs GPT partition tables, **`fdisk`** and **`parted`** for partition management
- **Filesystems** — what they are, the common types (**ext4**, **xfs**, **btrfs**, vfat, swap)
- **`mkfs`** — formatting a filesystem; **`fsck`** — checking and repairing one
- **Mounting** — `mount`, `umount`, what they actually do
- **`/etc/fstab`** — persistent mounts that come back after reboot
- **UUID and LABEL** — stable identifiers that survive device renames
- **`df` and `du`** — the two questions about disk space
- **Swap** — virtual memory backup; `mkswap`, `swapon`, `swappiness`
- **LVM** — Logical Volume Manager: PVs, VGs, LVs, snapshots, resizing — the big LFCS topic
- **RAID basics** — the common levels (0, 1, 5, 6, 10) and `mdadm`
- **Quotas** — limiting per-user disk usage

This notebook assumes notebooks 01-06. By the end you'll know how to inspect storage, create new filesystems, mount them durably, manage logical volumes, and reason about disk space and swap.

## The storage stack — a mental model

Before commands, the picture. When you `cat /etc/passwd`, the bytes travel through a stack of layers. Each layer turns one abstraction into a lower-level one:

```
        ┌─────────────────────────────────┐
        │  files & directories            │   "/etc/passwd"
        ├─────────────────────────────────┤
        │  filesystem (ext4, xfs, btrfs)  │   how bytes are organised
        ├─────────────────────────────────┤
        │  (optional) LVM logical volume  │   software volume management
        ├─────────────────────────────────┤
        │  (optional) RAID array          │   redundancy across disks
        ├─────────────────────────────────┤
        │  partition (e.g. /dev/sda2)     │   a slice of a disk
        ├─────────────────────────────────┤
        │  block device (/dev/sda)        │   the kernel's view of the disk
        ├─────────────────────────────────┤
        │  physical disk (SSD, HDD)       │   the hardware
        └─────────────────────────────────┘
```

A **simple** desktop setup: hardware → block device → one partition → ext4 filesystem → mounted at `/`.

A **production server** setup: hardware → multiple disks → RAID array → LVM volume group → multiple LVs → ext4/xfs filesystems → mounted at `/`, `/var`, `/home`.

This whole stack is composable. You can mix and match. The kernel doesn't care which layers you use; each layer just exposes a "block device" interface to the layer above it.

Work the stack **top-down** when you're inspecting (start at the mount, follow the layers down with `mount`/`lsblk`/`findmnt`). Work **bottom-up** when you're creating (partition → format → mount).

## Block devices and how Linux names them

A **block device** is the kernel's abstraction for any storage device — disks, SSDs, USB sticks, RAID arrays, LVM volumes — anything that reads and writes in fixed-size blocks (usually 4 KB). They all appear as files under `/dev/`.

Naming conventions you'll meet:

| Device pattern | What it is |
|---|---|
| **`/dev/sda`, `/dev/sdb`, ...** | SATA, SCSI, and USB disks. First letter "s", then `a`, `b`, `c` for first/second/third disk. |
| **`/dev/sda1`, `/dev/sda2`, ...** | Partitions on `/dev/sda` (the number is the partition index). |
| **`/dev/nvme0n1`** | First NVMe drive's first namespace (modern SSDs). |
| **`/dev/nvme0n1p1`, `/dev/nvme0n1p2`, ...** | Partitions on an NVMe drive (note the `p` before the number). |
| **`/dev/vda`, `/dev/vdb`, ...** | Virtio disks — common inside VMs (KVM, QEMU, cloud instances). |
| **`/dev/xvda`** | Xen virtual disks (older AWS EC2 instances). |
| **`/dev/sr0`** | First optical drive (CD/DVD). |
| **`/dev/loop0`** | Loop devices — let you mount a file as if it were a disk. |
| **`/dev/mapper/<name>`** | Device-mapper targets — LVM volumes, LUKS-encrypted devices. |
| **`/dev/md0`** | Software RAID arrays managed by `mdadm`. |

**Device names can change between boots** on systems with multiple disks. If you unplug the second disk and reboot, what was `/dev/sdb` might now be `/dev/sda`. That's why you should never reference disks by device name in `/etc/fstab` — use **UUID** or **LABEL** instead (covered below).

## `lsblk` — see what's attached

**`lsblk`** ("list block devices") is the first thing you run when you want to understand the storage on a system. It prints a tree showing every block device, its partitions, and where they're mounted:

```
NAME        MAJ:MIN RM   SIZE RO TYPE MOUNTPOINTS
sda           8:0    0   100G  0 disk
├─sda1        8:1    0   512M  0 part /boot/efi
├─sda2        8:2    0     1G  0 part /boot
└─sda3        8:3    0  98.5G  0 part
  └─ubuntu-root
            253:0    0  98.5G  0 lvm  /
```

Read top-down: the disk `sda`, its three partitions, and on partition 3 a logical volume `ubuntu-root` (notice the `lvm` type) that's mounted at `/`.

Useful flags:

| Flag | Effect |
|---|---|
| **`-f`** | also show filesystem type, label, UUID, and free space |
| **`-p`** | full device path (e.g. `/dev/sda` not `sda`) |
| **`-a`** | include empty devices |
| **`-J`** | output as JSON (useful for scripts) |
| **`-o COL1,COL2,...`** | choose columns explicitly (`lsblk --help` shows all options) |

Two related commands:

- **`blkid`** — print UUID and filesystem type for a device (or all devices if no argument).
- **`findmnt`** — show mounted filesystems as a tree, more readable than `mount` alone.

In [ ]:
!lsblk
!lsblk -f 2>/dev/null
!blkid 2>/dev/null | head -5

## Partitions and partition tables — MBR vs GPT

A **partition** is a contiguous slice of a disk with its own boundaries. Why partition?

- Multiple filesystems on one disk (one for `/`, one for `/home`, etc.)
- Reserved space for swap or recovery
- Multiple OSes dual-booting
- Smaller filesystems are faster to `fsck`

Partitions are described by a **partition table** at the start of the disk. Two formats exist:

**MBR (Master Boot Record)** — the old standard.
- Up to **4 primary partitions** (or 3 primary + 1 extended containing logicals).
- Maximum disk size **2 TB**.
- Stores bootloader code (legacy BIOS).
- Still everywhere on older systems.

**GPT (GUID Partition Table)** — the modern standard.
- Up to **128 partitions** by default.
- Practically no size limit (8 zettabytes).
- Built-in backup table at the end of the disk for redundancy.
- Required by UEFI firmware (most modern PCs and all modern servers).
- **Use GPT for any new disk** unless you have a specific reason not to.

You can tell which a disk uses with `parted` or `fdisk`:

```bash
sudo parted /dev/sda print
sudo fdisk -l /dev/sda
```

Look for "Partition Table: gpt" (parted) or "Disklabel type: gpt" (fdisk) in the output.

Two partitioning tools:

**`fdisk`** — classic, interactive. Used to be MBR-only; modern versions support GPT too. The interactive prompt walks you through creating partitions step by step. Common keys inside `fdisk`:
- `p` — print current partition table
- `n` — new partition
- `d` — delete a partition
- `t` — change a partition type (e.g. 82 = swap, 83 = Linux, fd = Linux RAID)
- `w` — write changes and exit
- `q` — quit without saving

**`parted`** — newer, supports both MBR and GPT, has a non-interactive command form (scriptable). Often preferred for GPT work. Example:

```bash
sudo parted /dev/sdb mklabel gpt              # create a GPT partition table
sudo parted /dev/sdb mkpart primary ext4 0% 50%   # create partition spanning 0-50% of disk
sudo parted /dev/sdb mkpart primary ext4 50% 100% # second partition spanning the rest
```

Also useful: **`gdisk`** — fdisk-like UI for GPT specifically; **`cfdisk`** — a friendlier ncurses TUI.

**Warning**: any partition change can destroy data. Always back up first. **Never partition a disk that has live data without a verified backup**, and always double-check the device path (`/dev/sda` vs `/dev/sdb`) before pressing write.

After writing partitions, run **`partprobe`** or **`udevadm settle`** to tell the kernel to re-read the partition table, then `lsblk` to verify.

In [ ]:
!sudo fdisk -l 2>/dev/null | head -20 || lsblk
!sudo parted -l 2>/dev/null | head -20 || lsblk

## Filesystems — the layer that turns blocks into files

A **partition** is just a slice of bytes. To store files in it, you need a **filesystem** — a layout that says "here's the index of files, here's where each file's data lives, here are the permissions". Without a filesystem, a partition is just empty space.

The common filesystems on Linux:

| Filesystem | Use when |
|---|---|
| **`ext4`** | the default on most distros — stable, mature, well-understood, good general purpose |
| **`xfs`** | RHEL/CentOS default; excellent for large files and high parallelism. Cannot shrink. |
| **`btrfs`** | "B-tree FS" — snapshots, subvolumes, built-in RAID. Default on openSUSE, Fedora workstation |
| **`vfat`** (FAT32) | universal compatibility (USB sticks, EFI system partitions). No permissions, no symlinks |
| **`exfat`** | FAT32's modern successor — large files, still no permissions. Common on SD cards |
| **`ntfs`** | Windows filesystem. Read/write supported via `ntfs-3g`. Useful for dual-boot |
| **`swap`** | not a real filesystem — a special area used as virtual memory. Created with `mkswap` |
| **`tmpfs`** | RAM-backed filesystem. Used for `/tmp`, `/run`. Vanishes on reboot |
| **`zfs`** | very capable but licensed differently — not in mainline kernel; available as an out-of-tree module on Ubuntu |

For LFCS, **know `ext4` deeply and `xfs` well**. The others are situational.

### Making a filesystem — `mkfs`

To format a partition with a filesystem, use `mkfs` with the type as a suffix:

```bash
sudo mkfs.ext4 /dev/sdb1                 # format /dev/sdb1 as ext4
sudo mkfs.xfs /dev/sdb1                  # format as XFS
sudo mkfs.vfat -F 32 /dev/sdb1           # FAT32 (note the -F 32 for the variant)
```

Useful options for `mkfs.ext4`:

- **`-L LABEL`** — set a filesystem label (a human-friendly name, max 16 chars)
- **`-m 1`** — reserve only 1% of space for root (default is 5% — wasted on big disks)
- **`-N 100000`** — preallocate this many inodes (for "lots of small files" workloads)
- **`-T largefile`** — use a preset suited to large files (uses fewer, bigger inodes)

**WARNING**: `mkfs` **destroys everything** on the target. There is no "format and keep existing data" option. The target must be a partition that's not currently mounted; `mkfs` will refuse otherwise unless forced.

### Checking and repairing — `fsck`

`fsck` checks a filesystem for inconsistencies. It's run **automatically at boot** for most mounted filesystems; you only run it manually after a crash, after suspected corruption, or as a precaution.

```bash
sudo fsck /dev/sdb1                      # check /dev/sdb1 (autodetects type)
sudo fsck.ext4 -y /dev/sdb1              # ext4 check, automatically answer yes to prompts
sudo fsck -N /dev/sdb1                   # dry run — just say what would be checked
```

**Critical rule: the filesystem must be UNMOUNTED before fsck.** Running `fsck` on a live filesystem can corrupt it. The boot-time fsck happens before the filesystem is mounted (or read-only mounted) for exactly this reason.

For XFS, `fsck.xfs` does nothing — XFS uses `xfs_repair` instead (and `xfs_check` for read-only inspection). For btrfs, the equivalent is `btrfs check`. Both must run on unmounted filesystems.

## Mounting — connecting filesystems to the tree

A filesystem isn't useful until it's **mounted** somewhere in the directory tree. **`mount`** attaches a filesystem at a directory; **`umount`** detaches it.

The full form:

```bash
sudo mount [-t TYPE] [-o OPTIONS] DEVICE MOUNT_POINT
```

Examples:

```bash
sudo mount /dev/sdb1 /mnt/data                       # mount at /mnt/data
sudo mount -t ext4 /dev/sdb1 /mnt/data               # explicit type (usually autodetected)
sudo mount -o ro /dev/sdb1 /mnt/data                 # read-only
sudo mount -o noatime,nodev /dev/sdb1 /mnt/data      # multiple options, comma-separated

sudo umount /mnt/data                                # unmount by mount point
sudo umount /dev/sdb1                                # ...or by device
```

The mount point must already exist as an empty directory. `mkdir /mnt/data` first if needed. Anything that was previously visible at `/mnt/data` gets hidden until you unmount.

Common mount options worth knowing:

| Option | Meaning |
|---|---|
| **`ro`** | read-only |
| **`rw`** | read-write (default) |
| **`noatime`** | don't update access times on read (performance) |
| **`nosuid`** | ignore SUID bits on this filesystem |
| **`nodev`** | ignore device nodes |
| **`noexec`** | don't allow execution of binaries |
| **`defaults`** | shorthand for `rw,suid,dev,exec,auto,nouser,async` |
| **`user`** | allow ordinary users to mount this entry (only meaningful in `fstab`) |

Three flavours of "what to mount" you'll see:

- **Block-backed** — a real device like `/dev/sdb1`.
- **Virtual** — `proc` on `/proc`, `sysfs` on `/sys`, `tmpfs` on `/tmp`. The "source" is a kernel filesystem type, not a device.
- **Bind mount** — `mount --bind /source /destination` — makes the same data appear at a second path. Heavily used by containers; covered briefly in notebook 03.

To see what's currently mounted:

```bash
mount                       # all mounts, verbose format
findmnt                     # tree view, cleaner output
findmnt --real              # only real (non-virtual) filesystems
df -hT                      # with usage and filesystem type
cat /proc/mounts            # raw kernel view (what's actually mounted)
```

`findmnt` is the friendliest of these. Use it.

In [ ]:
!findmnt --real 2>/dev/null | head -10 || mount | head -10
!df -hT 2>/dev/null | head -10

## `/etc/fstab` — persistent mounts

Mounts done with `mount` are temporary — they vanish on reboot. To make a mount persistent, add an entry to **`/etc/fstab`** ("file systems table").

Each line in `/etc/fstab` has **six space-separated fields**:

```
DEVICE                 MOUNT_POINT    TYPE     OPTIONS         DUMP    PASS
UUID=ab12-cd34-...     /              ext4     defaults        0       1
UUID=ef56-7890-...     /home          ext4     defaults        0       2
UUID=12345678-...      none           swap     sw              0       0
tmpfs                  /tmp           tmpfs    defaults,nosuid 0       0
/dev/sdb1              /mnt/data      ext4     noatime         0       2
```

Field by field:

1. **DEVICE** — what to mount. **Strongly prefer UUID** (covered next section). Device names like `/dev/sdb1` are unstable.
2. **MOUNT_POINT** — where to mount (must exist). `none` for swap.
3. **TYPE** — filesystem type (`ext4`, `xfs`, `swap`, `tmpfs`, etc.). `auto` to let the kernel guess.
4. **OPTIONS** — comma-separated mount options. `defaults` for normal use; `sw` for swap.
5. **DUMP** — used by the legacy `dump` backup tool. **Always set to `0` today** (dump is essentially unused).
6. **PASS** — order for boot-time `fsck`. `1` for the root filesystem, `2` for other filesystems, `0` for "don't check" (use for swap, tmpfs, network filesystems).

After editing `/etc/fstab`, test before rebooting:

```bash
sudo mount -a                # mount all entries in fstab that aren't already mounted
findmnt --verify             # validate fstab syntax (newer util-linux)
```

If `mount -a` errors, **fix it before rebooting**. A broken `fstab` can drop you into emergency mode at next boot.

**A common safety pattern**: when adding a new entry, mount it manually first to verify it works (`sudo mount /dev/sdb1 /mnt/data`), then `umount`, then add to `fstab`, then `sudo mount -a` to test the fstab entry, then reboot when confident.

## UUID and LABEL — the stable way to identify filesystems

Device names like `/dev/sdb` can change between boots. Use **UUIDs** instead. A UUID is a globally unique identifier assigned to a filesystem when it's created — and it doesn't change even if the device renames.

To see UUIDs and labels:

```bash
blkid                                    # all devices
blkid /dev/sda1                          # one device
lsblk -o NAME,FSTYPE,LABEL,UUID,SIZE     # in tabular form
```

Sample output:

```
/dev/sda1: UUID="ab12-cd34-ef56" TYPE="ext4" PARTUUID="12345678-01"
```

In `/etc/fstab`, use the UUID form:

```
UUID=ab12-cd34-ef56     /mnt/data     ext4    defaults    0    2
```

**Labels** are a human-friendly alternative. Set with `-L` at format time, or change later:

```bash
sudo mkfs.ext4 -L data /dev/sdb1                # at creation
sudo e2label /dev/sdb1 newlabel                 # change ext label later
sudo xfs_admin -L newlabel /dev/sdb1            # XFS equivalent
sudo fatlabel /dev/sdb1 NEW                     # vfat/exfat
```

In `/etc/fstab`, use LABEL:

```
LABEL=data    /mnt/data    ext4    defaults    0    2
```

**Which is better — UUID or LABEL?** UUID is the safe default — it's globally unique and never collides. Labels are easier to read in `lsblk` output but you have to keep them unique yourself. For LFCS, know both forms and that **device names like `/dev/sdb1` should not appear in fstab** in production.

In [ ]:
!blkid 2>/dev/null | head -10
!lsblk -o NAME,FSTYPE,LABEL,UUID,MOUNTPOINT 2>/dev/null

## `df` and `du` — the two questions about disk space

Two completely different questions, two completely different tools.

### `df` — "how much space is on my filesystems?"

`df` ("disk free") shows space usage **per mounted filesystem**:

```bash
df -h                  # human-readable sizes (GB, MB)
df -hT                 # also show filesystem type
df -i                  # show inode usage instead of bytes
df -h /home            # space on the filesystem containing /home
```

The columns:

- **Size** — total filesystem size
- **Used** — used space (counts the 5% root reserve on ext4!)
- **Avail** — space available to **non-root users** (already excludes the reserve)
- **Use%** — percentage used
- **Mounted on** — where it's mounted

**Two surprises in df output:**

1. **The 5% root reserve.** On ext4, by default 5% of the filesystem is reserved for root. If `df` says 95% used, you may already be unable to write as a regular user. Adjust with `tune2fs -m 1 /dev/sda1` to drop it to 1%.
2. **Inodes can run out before bytes.** Each file uses one inode, set at format time. Lots of tiny files on a small filesystem can exhaust inodes while you still have bytes free. `df -i` reveals this; "No space left on device" with bytes available is the giveaway.

### `du` — "what's using space inside this directory?"

`du` ("disk usage") walks a directory tree and tells you how much space each item takes:

```bash
du -sh /var/log                   # summary, human-readable, of the whole tree
du -h /var/log                    # every directory under /var/log
du -h --max-depth=1 /var/log      # one level deep — best for finding big subdirs
du -sh /var/log/*                 # everything inside /var/log, one summary per item
du -sh /var/log/* | sort -h       # sorted by size — the most useful form
```

The killer combo for **"find what's eating my disk"**:

```bash
du -sh /var/log/* 2>/dev/null | sort -h | tail -20
```

That gives you the top 20 space hogs under `/var/log`, sorted small-to-big.

**df vs du discrepancy.** Sometimes `df` says the disk is full but `du -sh /` shows much less usage. The classic cause: a deleted-but-still-open file. A process holds a file open; the file is `rm`'d, so it doesn't appear in `du`; but the kernel keeps the data around until the process closes the file. `lsof | grep deleted` finds these.

Modern alternatives worth installing:

- **`ncdu`** — interactive directory size browser (TUI). The fastest way to find big things.
- **`dust`** — colourised, sorted by default.

In [ ]:
!df -hT 2>/dev/null | head -10
!df -i 2>/dev/null | head -10
!du -sh /etc /var/log /tmp 2>/dev/null | sort -h

## Swap — virtual memory backup

**Swap** is disk space the kernel uses as overflow when RAM is full. Pages from RAM that haven't been touched recently can be written to swap, freeing RAM for active processes. When that page is needed again, the kernel reads it back in.

Swap can be either a **dedicated partition** or a **file**:

- **Swap partition** — set the partition type to `82` (Linux swap) when partitioning; common on older installs.
- **Swap file** — a regular file used as swap. More flexible; can be resized without re-partitioning. The default on many modern distros.

### Creating swap

For a swap **partition**:

```bash
sudo mkswap /dev/sdb2                    # format the partition as swap
sudo swapon /dev/sdb2                    # enable it now
```

For a swap **file**:

```bash
sudo fallocate -l 2G /swapfile           # create a 2 GB file (fast, sparse-aware)
sudo chmod 600 /swapfile                 # MUST be root-only readable
sudo mkswap /swapfile                    # format as swap
sudo swapon /swapfile                    # enable now
```

To check active swap:

```bash
swapon --show
free -h                                  # shows Mem and Swap totals
cat /proc/swaps                          # the kernel's raw view
```

To disable:

```bash
sudo swapoff /swapfile                   # disable a specific swap
sudo swapoff -a                          # disable all swap
```

For persistence, add to `/etc/fstab`:

```
/swapfile    none    swap    sw    0    0
```

After editing, `sudo swapon -a` to enable everything in fstab.

### Swappiness

`vm.swappiness` controls how aggressively the kernel uses swap. Range 0-200; default is **60**. Lower = use swap less (favour RAM, evict cache); higher = use swap more.

```bash
cat /proc/sys/vm/swappiness              # current value
sudo sysctl vm.swappiness=10             # set temporarily
echo 'vm.swappiness=10' | sudo tee -a /etc/sysctl.conf    # persistent
```

**Recommended values:**

- **`60`** — desktop default; balanced
- **`10-20`** — server default; favour keeping things in RAM
- **`100+`** — exotic; useful only when swap is very fast (e.g. NVMe-backed)
- **`0`** — almost never; only swap if RAM is completely exhausted (risk of OOM kills before swap is used)

**Modern note**: on systems with lots of RAM, swap is less critical than it used to be. But you usually still want some — hibernate needs swap of at least RAM size, and a small swap acts as a pressure release valve. Cloud VMs often ship without swap; adding a swap file is a common first tweak.

In [ ]:
!swapon --show 2>/dev/null
!free -h
!cat /proc/sys/vm/swappiness

## LVM — flexible storage management

**LVM (Logical Volume Manager)** sits between partitions and filesystems. It lets you treat multiple physical disks (or partitions) as one pool, then carve out **logical volumes** from that pool — and resize them on the fly without losing data.

The three-layer model:

```
  Physical Volumes (PVs)          ← /dev/sda2, /dev/sdb1, /dev/sdc1 (real partitions/disks)
        ↓ aggregated into
  Volume Group (VG)               ← one big pool of storage, e.g. "ubuntu-vg"
        ↓ carved into
  Logical Volumes (LVs)           ← "root-lv", "home-lv", "var-lv" (act like partitions)
        ↓ formatted with
  Filesystems                     ← ext4, xfs on top of each LV
```

The killer features:

- **Resize LVs without unmounting**: grow them while in use.
- **Add disks to expand the pool**: physical volumes can be added to an existing VG.
- **Snapshots**: instant point-in-time copies of an LV — for backups or risky operations.
- **Span filesystems across multiple disks** without RAID.
- **Move data off a disk** while online (`pvmove`) — useful for replacing a failing drive.

### Creating LVM — the standard workflow

```bash
# 1. Mark partitions as Physical Volumes
sudo pvcreate /dev/sdb1 /dev/sdc1

# 2. Combine PVs into a Volume Group
sudo vgcreate datavg /dev/sdb1 /dev/sdc1

# 3. Carve out Logical Volumes
sudo lvcreate -L 20G -n appdata datavg          # 20 GB LV named "appdata"
sudo lvcreate -L 10G -n logs datavg             # 10 GB LV named "logs"
sudo lvcreate -l 100%FREE -n bigdata datavg     # use all remaining space

# 4. Format the LVs with a filesystem
sudo mkfs.ext4 /dev/datavg/appdata
sudo mkfs.ext4 /dev/datavg/logs

# 5. Mount them
sudo mkdir -p /srv/app /srv/logs
sudo mount /dev/datavg/appdata /srv/app
sudo mount /dev/datavg/logs /srv/logs

# 6. Add to /etc/fstab for persistence (use UUID via blkid)
```

LV device paths show up as **`/dev/VG_NAME/LV_NAME`** and also as **`/dev/mapper/VG_NAME-LV_NAME`** (the canonical kernel name).

### Inspecting LVM

Three "show me what's there" commands at each level:

```bash
pvs                              # one-line PV summary
pvdisplay                        # detailed PV info

vgs                              # one-line VG summary
vgdisplay                        # detailed VG info

lvs                              # one-line LV summary
lvdisplay                        # detailed LV info
```

## LVM operations — resize, snapshots, extend

### Resizing a Logical Volume

The whole reason LVM exists. Two steps: **resize the LV**, then **resize the filesystem on it**.

To **grow** an LV (and the filesystem on it), in one command:

```bash
sudo lvextend -L +5G -r /dev/datavg/appdata     # +5G, and -r resizes the FS at the same time
sudo lvextend -L 50G -r /dev/datavg/appdata     # set absolute size
sudo lvextend -l +100%FREE -r /dev/datavg/appdata    # take all remaining VG space
```

The `-r` (or `--resizefs`) flag is the killer: it grows the underlying filesystem (`ext4`, `xfs`, etc.) for you in the same call. Without `-r`, you have to call `resize2fs` (ext) or `xfs_growfs` (xfs) yourself.

To **shrink** an LV — **only ext4 supports shrinking; XFS cannot shrink**. For ext4:

```bash
sudo umount /srv/app
sudo e2fsck -f /dev/datavg/appdata               # required before shrinking
sudo resize2fs /dev/datavg/appdata 10G           # shrink the filesystem first
sudo lvreduce -L 10G /dev/datavg/appdata         # then shrink the LV
sudo mount /srv/app
```

Shrinking is risky — back up first. **Growing is the LVM superpower; do that whenever possible**.

### Extending the Volume Group (adding more disks)

When you run out of space in the pool, add another PV:

```bash
sudo pvcreate /dev/sdd1                          # prep the new partition
sudo vgextend datavg /dev/sdd1                   # add it to the VG
sudo vgs                                         # verify increased VFree
```

Now you have more free space in `datavg` to grow LVs with.

### Snapshots

A **snapshot** is an instant, space-efficient point-in-time copy of an LV. Useful for:

- Taking a consistent backup while the original stays in use
- Trying a risky operation (you can roll back if it goes wrong)

```bash
# Create a snapshot with 5 GB of "change tracking" space
sudo lvcreate -L 5G -s -n appdata-snap /dev/datavg/appdata

# The snapshot is itself a block device — you can mount it read-only to back it up
sudo mkdir /mnt/snap
sudo mount -o ro /dev/datavg/appdata-snap /mnt/snap
# ... tar/rsync from /mnt/snap ...
sudo umount /mnt/snap

# Done — remove the snapshot
sudo lvremove /dev/datavg/appdata-snap
```

**Snapshot size sizing**: a snapshot tracks **changes** made to the original after the snapshot. Size it as big as the expected churn. If the snapshot fills up, it becomes invalid (LVM marks it as such). For backups, a few hours of churn is usually plenty.

### Moving data off a disk (`pvmove`)

If a disk is failing or you want to retire it, `pvmove` migrates its data to other PVs in the same VG, **while everything stays online**:

```bash
sudo pvmove /dev/sdb1                            # move all extents off sdb1
sudo vgreduce datavg /dev/sdb1                   # remove sdb1 from the VG
sudo pvremove /dev/sdb1                          # remove PV signature
# Now you can pull the disk
```

For LFCS, you should be able to **create a PV/VG/LV from scratch, extend a VG with a new disk, grow an LV (with `-r`), and take a snapshot**.

In [ ]:
!sudo pvs 2>/dev/null
!sudo vgs 2>/dev/null
!sudo lvs 2>/dev/null

## RAID basics — redundancy across disks

**RAID** ("Redundant Array of Independent Disks") combines multiple physical disks into one logical device with redundancy, performance, or both. Linux supports **software RAID** via the `mdadm` tool (the kernel's `md` subsystem) — independent of any hardware RAID controller.

The common RAID levels:

| Level | What it does | Disks needed | Capacity | Speed | Survives |
|---|---|---|---|---|---|
| **RAID 0** (stripe) | data split across disks | 2+ | sum of all | fastest reads/writes | **nothing** — one disk dies, all data lost |
| **RAID 1** (mirror) | identical copy on every disk | 2+ | smallest disk | reads fast, writes normal | up to N-1 disk failures |
| **RAID 5** | striping + 1 parity block | 3+ | (N-1) × smallest | good reads, slow writes | 1 disk failure |
| **RAID 6** | striping + 2 parity blocks | 4+ | (N-2) × smallest | good reads, slow writes | 2 disk failures |
| **RAID 10** | mirror + stripe | 4+ (even) | half the total | fast both ways | up to half the disks (1 per mirror pair) |

**Picking a level:**

- **RAID 0**: only when speed matters more than data (caches, scratch). Risky.
- **RAID 1**: simplest redundancy. Two disks; lose one, keep going. Common for boot drives.
- **RAID 5**: economical (only 1 disk of overhead) but **slow on writes** and risky to rebuild on large modern disks (rebuilds can take days, during which another failure kills the array). **Not recommended for new builds** with disks > ~4 TB.
- **RAID 6**: like RAID 5 but tolerates two failures. The safer choice when capacity matters.
- **RAID 10**: best balance of speed + reliability if you can afford halving capacity. The right default for performance-sensitive arrays.

**RAID is not a backup.** RAID protects against disk *failure*. It does not protect against `rm -rf`, ransomware, fire, theft, or filesystem corruption — all of those happily replicate to every disk in the array. Always run separate backups.

### `mdadm` — creating and managing arrays

Create a RAID array:

```bash
# RAID 1 mirror across two disks
sudo mdadm --create /dev/md0 --level=1 --raid-devices=2 /dev/sdb1 /dev/sdc1

# RAID 10 across four disks
sudo mdadm --create /dev/md0 --level=10 --raid-devices=4 \
    /dev/sdb1 /dev/sdc1 /dev/sdd1 /dev/sde1
```

After creation, the array is a single block device `/dev/md0` you can format and mount:

```bash
sudo mkfs.ext4 /dev/md0
sudo mount /dev/md0 /srv/storage
```

Inspect arrays:

```bash
cat /proc/mdstat                          # quick status of all arrays
sudo mdadm --detail /dev/md0              # detailed info, including disk states
```

Handle a failed disk:

```bash
sudo mdadm --manage /dev/md0 --fail /dev/sdb1      # mark as failed (for testing)
sudo mdadm --manage /dev/md0 --remove /dev/sdb1    # remove from array
sudo mdadm --manage /dev/md0 --add /dev/sdf1       # add a replacement
# Watch the rebuild with: cat /proc/mdstat
```

Make the array persistent by saving its config:

```bash
sudo mdadm --detail --scan | sudo tee -a /etc/mdadm/mdadm.conf
sudo update-initramfs -u                  # so the kernel can assemble the array at boot
```

For LFCS-level work: **know the common RAID levels, when to choose each, and how to create/inspect/repair an array with `mdadm`**.

## Quotas — per-user limits on disk usage

Quotas let you limit how much disk space a user (or group) can use on a filesystem. Useful on multi-user systems (shared servers, web hosting, lab machines) to prevent any one user from filling the disk.

There are two kinds of limits:

- **Block quota** — limits total bytes used.
- **Inode quota** — limits number of files created.

And two levels of strictness:

- **Soft limit** — a warning threshold; users can exceed it temporarily during a grace period (default 7 days).
- **Hard limit** — an absolute ceiling; writes fail past this.

### Setup workflow (one-time, per filesystem)

1. **Install quota tools**:

```bash
sudo apt install quota                   # Debian/Ubuntu
sudo dnf install quota                   # RHEL/Fedora
```

2. **Enable quotas on the target filesystem in `/etc/fstab`** — add `usrquota,grpquota` to the mount options:

```
UUID=...    /home    ext4    defaults,usrquota,grpquota    0    2
```

3. **Remount and initialise** the quota databases:

```bash
sudo mount -o remount /home
sudo quotacheck -cugm /home              # create user and group quota files
sudo quotaon /home                       # turn quotas on
```

### Setting and viewing quotas

Edit a user's quota interactively:

```bash
sudo edquota -u alice                    # opens $EDITOR with alice's quotas
sudo edquota -g devs                     # group quota
```

You'll see soft/hard limits for blocks and inodes. Set them in the editor and save.

Set non-interactively (scriptable):

```bash
sudo setquota -u alice 100000 110000 1000 1100 /home
#              user   bsoft  bhard  isoft ihard filesystem
# Block limits in 1KB blocks: 100 MB soft, 110 MB hard
# Inode limits: 1000 soft, 1100 hard
```

Inspect quotas:

```bash
sudo quota -u alice                      # show alice's quotas + usage
sudo repquota -a                         # report all users' quotas, all filesystems
```

Change the grace period (how long users can be over the soft limit):

```bash
sudo edquota -t                          # opens editor; defaults: 7 days
```

For LFCS: know how to **enable quotas in `/etc/fstab`, initialise with `quotacheck`/`quotaon`, set limits with `edquota` or `setquota`, and inspect with `repquota`**.

On `xfs`, quotas work slightly differently — enabled via mount option `uquota`/`gquota` and managed with `xfs_quota`. Same concepts, different tool.

## What you've learned

A compact recap to keep next to your prompt:

- **The storage stack** — hardware → block device → partition → (optional RAID) → (optional LVM) → filesystem → mount point.
- **Block device names** — `/dev/sda` (SATA/SCSI), `/dev/nvme0n1` (NVMe), `/dev/vda` (virtio/VM). Numbers append for partitions (`/dev/sda1`, `/dev/nvme0n1p1`).
- **`lsblk`** — the first command to run. **`blkid`** for UUIDs. **`findmnt`** for mounts.
- **Partitions** — MBR (legacy, ≤2 TB, ≤4 primary partitions) vs GPT (modern default, up to 128 partitions, no practical size limit). Tools: **`fdisk`** (interactive), **`parted`** (scriptable), **`gdisk`** (GPT-focused).
- **Filesystems** — **`ext4`** (general default), **`xfs`** (RHEL default, can't shrink), `btrfs` (snapshots built-in), `vfat`/`exfat` (cross-platform), `swap`, `tmpfs`. Format with **`mkfs.<type>`**.
- **`fsck`** — only on **unmounted** filesystems. XFS uses `xfs_repair` instead.
- **Mounting** — `mount [-o options] DEVICE POINT`, `umount`. Common options: `ro`, `noatime`, `noexec`, `nosuid`, `nodev`. **`findmnt`** for the tree view.
- **`/etc/fstab`** — 6 fields: device, mount-point, type, options, dump (0), pass (1 for /, 2 for others, 0 for skip). Test with `mount -a` before reboot.
- **UUID over device names** — get from `blkid`. Survives device renames.
- **`df -h`** for filesystem usage. **`df -i`** for inode usage. **`du -sh /var/log/* | sort -h | tail`** for finding space hogs.
- **Swap** — `mkswap`, `swapon`, `swapoff`. Set `vm.swappiness` (default 60; 10-20 for servers). Add a swap file with `fallocate -l 2G /swapfile && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile`.
- **LVM** — `pvcreate` → `vgcreate` → `lvcreate`. Inspect with `pvs`/`vgs`/`lvs`. **Grow LVs with `lvextend -L +5G -r LV`** (the `-r` resizes the filesystem too). Add disks with `vgextend`. Snapshots with `lvcreate -s`. Move data with `pvmove`.
- **RAID** — `mdadm --create /dev/md0 --level=N --raid-devices=K disk...`. Common levels: **1** (mirror), **6** (2-disk-failure tolerance), **10** (mirror+stripe). Save config to `/etc/mdadm/mdadm.conf`. Status in `/proc/mdstat`.
- **Quotas** — add `usrquota,grpquota` to fstab; `quotacheck -cugm`; `quotaon`; `edquota -u USER`; `repquota -a` to inspect.
- **RAID is not a backup.** Always keep separate, off-system backups.

**Practise before moving on.** On your Linux VM, attach a small extra disk (or create a loop device from a file). Walk the bottom-up workflow: `parted` to create a partition table and a partition, `mkfs.ext4` to format, `mount` to attach, then add to `/etc/fstab` using UUID. If you can, build a small LVM setup: `pvcreate` on two loop devices, `vgcreate`, `lvcreate`, `lvextend -L +500M -r`. Take a snapshot, mount it, remove it. Hands-on time pays back enormously here.

**Next:** notebook `08-networking-and-ssh` moves from disks to the wire — IP addresses, interfaces, routing, DNS, SSH (including keys and config), firewalls, and network troubleshooting.